# Stability Runs Analysis (v0.2-alpha)

This notebook analyzes repeated retrieval-evaluation runs and visualizes:

- latency and QPS stability over run index
- distribution of latency and QPS
- quality metric consistency (`precision@k`, `recall@k`, `ndcg@k`)

Expected inputs:

- `artifacts/testing_runs/stability_runs_bruteforce_200.jsonl`
- `artifacts/testing_runs/stability_summary_bruteforce_200.json`

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RUNS_PATH = Path("artifacts/testing_runs/stability_runs_bruteforce_200.jsonl")
SUMMARY_PATH = Path("artifacts/testing_runs/stability_summary_bruteforce_200.json")

rows = [json.loads(line) for line in RUNS_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))

df = pd.DataFrame(
    {
        "run_index": [r["run_index"] for r in rows],
        "latency_p50_ms": [r["performance"]["latency_p50_ms"] for r in rows],
        "latency_p95_ms": [r["performance"]["latency_p95_ms"] for r in rows],
        "qps": [r["performance"]["qps"] for r in rows],
        "recall_at_max_k": [r["metrics"][f"recall@{max(summary['config']['ks'])}"] for r in rows],
        "ndcg_at_max_k": [r["metrics"][f"ndcg@{max(summary['config']['ks'])}"] for r in rows],
    }
)

df.head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(df["run_index"], df["latency_p50_ms"], linewidth=1.2)
axes[0, 0].set_title("Latency p50 over runs")
axes[0, 0].set_xlabel("run")
axes[0, 0].set_ylabel("ms")

axes[0, 1].plot(df["run_index"], df["latency_p95_ms"], linewidth=1.2, color="tab:orange")
axes[0, 1].set_title("Latency p95 over runs")
axes[0, 1].set_xlabel("run")
axes[0, 1].set_ylabel("ms")

axes[1, 0].plot(df["run_index"], df["qps"], linewidth=1.2, color="tab:green")
axes[1, 0].set_title("QPS over runs")
axes[1, 0].set_xlabel("run")
axes[1, 0].set_ylabel("qps")

axes[1, 1].hist(df["latency_p95_ms"], bins=20, alpha=0.8, color="tab:red")
axes[1, 1].set_title("Latency p95 distribution")
axes[1, 1].set_xlabel("ms")
axes[1, 1].set_ylabel("count")

plt.tight_layout()
plt.show()

In [ ]:
def describe_series(s: pd.Series) -> dict[str, float]:
    arr = s.to_numpy(dtype=float)
    mean = float(np.mean(arr))
    std = float(np.std(arr))
    return {
        "mean": mean,
        "median": float(np.median(arr)),
        "std": std,
        "cv": float(std / mean) if mean != 0.0 else 0.0,
        "p02_5": float(np.percentile(arr, 2.5)),
        "p97_5": float(np.percentile(arr, 97.5)),
    }

report = {
    "run_count": int(df.shape[0]),
    "latency_p50_ms": describe_series(df["latency_p50_ms"]),
    "latency_p95_ms": describe_series(df["latency_p95_ms"]),
    "qps": describe_series(df["qps"]),
    "recall_at_max_k": describe_series(df["recall_at_max_k"]),
    "ndcg_at_max_k": describe_series(df["ndcg_at_max_k"]),
}

report

In [ ]:
# Optional: save a lightweight plot artifact for docs/release notes.
out_png = Path("artifacts/testing_runs/stability_plot_p95_qps.png")
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].hist(df["latency_p95_ms"], bins=20)
ax[0].set_title("p95 latency distribution")
ax[0].set_xlabel("ms")
ax[1].hist(df["qps"], bins=20)
ax[1].set_title("QPS distribution")
ax[1].set_xlabel("qps")
plt.tight_layout()
plt.savefig(out_png, dpi=140)
out_png